# Khadoom AI — Qwen 3 Open-Weight Inference Server (Colab L4 GPU)

Run **Qwen 3** on a **Google Colab NVIDIA L4 (24 GB VRAM)** with an OpenAI-compatible **vLLM** server, XGrammar structured tool calling, and a zero-signup **Cloudflare Quick Tunnel**.

### Setup Instructions
1. Open **Runtime → Change runtime type** and select **L4 GPU** (24 GB VRAM).
2. Run Cells 1–3 sequentially to start `vllm serve` + `cloudflared`.
3. Copy the printed `.env` block from Cell 4 into `khadoom-agent-harness/.env` locally.
4. Run `npm run server` or `npm start` in `khadoom-agent-harness` — Khadoom connects straight to Colab!

In [1]:
#@title 1. Check L4 GPU & Install Dependencies (Python 3.12 venv for vLLM CUDA wheels)
!nvidia-smi || (echo 'ERROR: No GPU detected! Go to Runtime -> Change runtime type -> L4 GPU.' && exit 1)

!pip install -q uv openai
!uv venv /opt/vllm-venv --python 3.12 --clear
!uv pip install --python /opt/vllm-venv/bin/python vllm --torch-backend=auto fastapi uvicorn sentence-transformers
!/opt/vllm-venv/bin/python -c "import torch, vllm, vllm._C; assert torch.cuda.is_available(), 'CUDA unavailable'; print('vLLM CUDA OK:', vllm.__version__, '| GPU:', torch.cuda.get_device_name(0))"

!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

Sat Nov 28 20:13:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   49C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#@title 2. Launch vLLM Server with Qwen 3 & XGrammar Tool Parsing
import subprocess, time, urllib.request, json

MODEL_ID = "Qwen/Qwen3-8B-FP8"  #@param ["Qwen/Qwen3-8B-FP8", "Qwen/Qwen3-14B-AWQ", "Qwen/Qwen3-8B", "Qwen/Qwen3-8B-AWQ"]
MAX_MODEL_LEN = 8192           #@param {type:"integer"}
GPU_MEM_UTIL = 0.85            #@param {type:"number"}
TOOL_PARSER = "hermes"      #@param ["hermes", "qwen3_xml"]

# Kill any stale vllm processes
!pkill -f "vllm serve" || true

vllm_cmd = [
    "vllm", "serve", MODEL_ID,
    "--host", "127.0.0.1",
    "--port", "8000",
    "--tensor-parallel-size", "1",
    "--gpu-memory-utilization", str(GPU_MEM_UTIL),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--enable-auto-tool-choice",
    "--tool-call-parser", TOOL_PARSER,
    "--reasoning-parser", "qwen3",
]

log_file = open("/tmp/vllm.log", "w")
vllm_proc = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=subprocess.STDOUT)
print(f"Starting {MODEL_ID} on port 8000 (first run downloads weights ~2-4 min)...")

ready = False
for i in range(180):
    if vllm_proc.poll() is not None:
        print("vLLM exited unexpectedly! Last 120 log lines:")
        !tail -n 120 /tmp/vllm.log
        raise RuntimeError("vLLM startup failed")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2) as resp:
            data = json.loads(resp.read().decode())
            print(f"vLLM READY after {i*2}s! Models:", [m["id"] for m in data.get("data", [])])
            ready = True
            break
    except Exception:
        time.sleep(2)

if not ready:
    !tail -n 40 /tmp/vllm.log
    raise TimeoutError("Timed out waiting for vLLM server")

^C
Starting Qwen/Qwen3-8B-FP8 on port 8000 (first run downloads weights ~2-4 min)...


In [ ]:
#@title 3. (Optional) Start Lightweight OpenAI-Compatible Embeddings & Reranker Sidecar (Port 8001)
#@markdown Hosts Qwen3-Embedding-0.6B (/v1/embeddings) AND BAAI/bge-reranker-v2-m3 (/v1/rerank) on the same L4 (~2.5 GB VRAM).
START_EMBEDDING_SIDECAR = True   #@param {type:"boolean"}
EMBEDDING_MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"  #@param ["Qwen/Qwen3-Embedding-0.6B", "BAAI/bge-m3"]
START_RERANKER = True            #@param {type:"boolean"}
RERANKER_MODEL_ID = "BAAI/bge-reranker-v2-m3"   #@param ["BAAI/bge-reranker-v2-m3"]

!pkill -f "embed_sidecar.py" || true

if START_EMBEDDING_SIDECAR:
    sidecar_code = f'''
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Union, List
from sentence_transformers import SentenceTransformer, CrossEncoder
import uvicorn

app = FastAPI()
model = SentenceTransformer("{EMBEDDING_MODEL_ID}", device="cuda")
reranker = CrossEncoder("{RERANKER_MODEL_ID}", device="cuda") if {START_RERANKER} else None

class EmbedRequest(BaseModel):
    model: str | None = None
    input: Union[str, List[str]]

class RerankRequest(BaseModel):
    model: str | None = None
    query: str
    documents: List[str]
    top_n: int | None = None

@app.get("/health")
def health():
    return {{
        "status": "ok",
        "embeddingModel": "{EMBEDDING_MODEL_ID}",
        "rerankerEnabled": reranker is not None,
        "rerankerModel": "{RERANKER_MODEL_ID}" if reranker is not None else None,
    }}

@app.post("/v1/embeddings")
def create_embeddings(req: EmbedRequest):
    texts = [req.input] if isinstance(req.input, str) else req.input
    vecs = model.encode(texts, normalize_embeddings=True).tolist()
    return {{
        "object": "list",
        "data": [{{"object": "embedding", "index": i, "embedding": v}} for i, v in enumerate(vecs)],
        "model": "{EMBEDDING_MODEL_ID}",
        "usage": {{"prompt_tokens": 0, "total_tokens": 0}}
    }}

@app.post("/v1/rerank")
def rerank_documents(req: RerankRequest):
    if reranker is None:
        return {{"results": []}}
    pairs = [(req.query, doc) for doc in req.documents]
    scores = reranker.predict(pairs).tolist()
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    limit = req.top_n or len(ranked)
    return {{
        "model": "{RERANKER_MODEL_ID}",
        "results": [{{"index": idx, "relevance_score": float(score)}} for idx, score in ranked[:limit]],
    }}

uvicorn.run(app, host="127.0.0.1", port=8001, log_level="warning")
'''
    with open("/tmp/embed_sidecar.py", "w") as f:
        f.write(sidecar_code)
    subprocess.Popen(["python3", "/tmp/embed_sidecar.py"], stdout=open("/tmp/embed.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(90):
        try:
            with urllib.request.urlopen("http://127.0.0.1:8001/health", timeout=2) as r:
                print(f"Sidecar ({EMBEDDING_MODEL_ID} + {RERANKER_MODEL_ID if START_RERANKER else 'no reranker'}) READY on port 8001!")
                break
        except Exception:
            time.sleep(2)
else:
    print("Skipping embeddings & reranker sidecar.")

In [ ]:
#@title 4. Open Cloudflare Tunnel & Print Khadoom `.env` Snippet
import re, subprocess, time

!pkill -f "cloudflared tunnel" || true

def start_tunnel(port, logfile):
    f = open(logfile, "w")
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=f, stderr=subprocess.STDOUT
    )
    url = None
    for _ in range(30):
        text = open(logfile).read()
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
        if m:
            url = m.group(0)
            break
        time.sleep(1)
    return proc, url

llm_tunnel_proc, llm_url = start_tunnel(8000, "/tmp/cf_llm.log")
if not llm_url:
    !tail -n 30 /tmp/cf_llm.log
    raise RuntimeError("Could not start Cloudflare tunnel for port 8000")

embed_url = None
if START_EMBEDDING_SIDECAR:
    embed_tunnel_proc, embed_url = start_tunnel(8001, "/tmp/cf_embed.log")

print("=" * 60)
print("Copy & paste this block into khadoom-agent-harness/.env:")
print("=" * 60)
print(f"LLM_BASE_URL={llm_url}/v1")
print("LLM_API_KEY=not-needed")
print(f"LLM_MODEL={MODEL_ID}")
print("LLM_DISABLE_THINKING=true")
print("LLM_HOURLY_COST_USD=0.25")
if embed_url:
    print(f"EMBEDDING_BASE_URL={embed_url}/v1")
    print("EMBEDDING_API_KEY=not-needed")
    print(f"EMBEDDING_MODEL={EMBEDDING_MODEL_ID}")
    if START_RERANKER:
        print(f"RERANKER_BASE_URL={embed_url}/v1")
        print(f"RERANKER_MODEL={RERANKER_MODEL_ID}")
    print("# Remember to run: npm run embed")
print("=" * 60)

In [ ]:
#@title 5. Local Tool-Calling & JSON Strictness Benchmark inside Colab
import time, json
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="not-needed")

tools = [{
    "type": "function",
    "function": {
        "name": "propose_store_changes",
        "description": "Propose RFC 6902 JSON Patches for a store",
        "parameters": {
            "type": "object",
            "properties": {
                "explanation": {"type": "string"},
                "patches": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "op": {"type": "string", "enum": ["add", "remove", "replace", "increment", "decrement", "percentage", "multiply"]},
                            "path": {"type": "string"},
                            "value": {}
                        },
                        "required": ["op", "path"]
                    }
                }
            },
            "required": ["explanation", "patches"]
        }
    }
}]

t0 = time.time()
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "Use propose_store_changes with schema paths like /paymentProviders[id=tabby]/enabled and /theme/colors/primary."},
        {"role": "user", "content": "Enable Tabby installments and set primary color to #8B5CF6"}
    ],
    tools=tools,
    tool_choice="auto",
    temperature=0.1,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
latency_ms = round((time.time() - t0) * 1000)

msg = resp.choices[0].message
out_tok = resp.usage.completion_tokens
tok_s = round(out_tok / (latency_ms / 1000), 2) if latency_ms > 0 else 0
ms_per_tok = round(latency_ms / out_tok, 2) if out_tok > 0 else 0
print(f"Latency: {latency_ms} ms | Speed: {tok_s} tok/s ({ms_per_tok} ms/out-tok) | Tokens: {resp.usage.prompt_tokens} in / {out_tok} out")
print(f"Estimated L4 GPU share (@ $0.25/hr): ${(latency_ms / 3_600_000 * 0.25):.6f}")
if msg.tool_calls:
    parsed = json.loads(msg.tool_calls[0].function.arguments)
    print("Parsed tool JSON cleanly:")
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
else:
    print("Model returned text instead of tool call:", msg.content)